In [ ]:
# importing functions
import pandas as pd
import chromadb

from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [ ]:
# Get the project root directory, regardless of where the notebook is running.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

# Load the processed Huntress dataset from the JSON file.
df = pd.read_json(root / "data/processed/huntress.json")

# Display the dataset to verify it was loaded correctly.
display(df)

,character,category,topic,text,source_url
0,The Huntress,power,Hunting Hatchets,"Begin the trial with 5 Hunting Hatchets, which...",https://deadbydaylight.com/game/characters/the...
1,The Huntress,perk,Beast of Prey,Lose your Terror Radius and Red Stain upon ent...,https://deadbydaylight.com/game/characters/the...
2,The Huntress,perk,Territorial Imperative,"When at a distance, view the aura of any Survi...",https://deadbydaylight.com/game/characters/the...
3,The Huntress,perk,Hex: Huntress Lullaby,Spawn a Hex Totem. Receive 1 token upon hookin...,https://deadbydaylight.com/game/characters/the...
4,The Huntress,lore,The Huntress’ Lore,"After her mother died hunting an Elk, Anna dis...",https://deadbydaylight.com/game/characters/the...


We use LangChain's RecursiveCharacterTextSplitter to divide text into smaller chunks while preserving natural text boundaries.

- **Chunk size (800):** Keeps each chunk manageable while retaining sufficient context.
- **Chunk overlap (100):** Helps prevent information loss between consecutive chunks.

**Metadata:** Preserves character names, categories, topics, and source URLs for easier identification and retrieval.

This prepares the text for embedding generation and semantic search. Since our descriptions are relatively short, most remain as individual chunks.

These parameters were selected as reasonable starting values rather than experimentally optimised settings.

In [ ]:
# Configure the text splitter to divide long descriptions into smaller chunks.
# Overlap helps preserve context between consecutive chunks.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = []

# Process each record and split its description into chunks.
for _, row in df.iterrows():
    for i, text in enumerate(splitter.split_text(row["text"])):
        chunks.append({
            "character": row["character"],
            "category": row["category"],
            "topic": row["topic"],
            "chunk_id": i,
            "text": f"{row['character']} - {row['topic']}\n{text}",
            "source_url": row["source_url"]
        })

# Convert the chunks into a DataFrame for embedding generation.
chunks_df = pd.DataFrame(chunks)

# Display the converted data frame.
display(chunks_df)

,character,category,topic,chunk_id,text,source_url
0,The Huntress,power,Hunting Hatchets,0,The Huntress - Hunting Hatchets\nBegin the tri...,https://deadbydaylight.com/game/characters/the...
1,The Huntress,perk,Beast of Prey,0,The Huntress - Beast of Prey\nLose your Terror...,https://deadbydaylight.com/game/characters/the...
2,The Huntress,perk,Territorial Imperative,0,The Huntress - Territorial Imperative\nWhen at...,https://deadbydaylight.com/game/characters/the...
3,The Huntress,perk,Hex: Huntress Lullaby,0,The Huntress - Hex: Huntress Lullaby\nSpawn a ...,https://deadbydaylight.com/game/characters/the...
4,The Huntress,lore,The Huntress’ Lore,0,The Huntress - The Huntress’ Lore\nAfter her m...,https://deadbydaylight.com/game/characters/the...


In [7]:
print("Total records:", len(df))
print("Total chunks:", len(chunks_df))
print("Shortest chunk:", chunks_df["text"].str.len().min())
print("Longest chunk:", chunks_df["text"].str.len().max())

display(chunks_df.groupby("category").size())

Total records: 5
Total chunks: 5
Shortest chunk: 146
Longest chunk: 546


category
lore     1
perk     3
power    1
dtype: int64

In [8]:
chunks_df["text_length"] = chunks_df["text"].str.len()
display(chunks_df[["category", "topic", "text_length"]])

,category,topic,text_length
0,power,Hunting Hatchets,170
1,perk,Beast of Prey,146
2,perk,Territorial Imperative,164
3,perk,Hex: Huntress Lullaby,546
4,lore,The Huntress’ Lore,521


We use Sentence Transformers' all-MiniLM-L6-v2 model to convert text chunks into 384-dimensional numerical vectors.

**Why this model?** It is lightweight, efficient, and suitable for semantic similarity tasks.

**Why embeddings?** They allow the system to retrieve information based on semantic meaning rather than exact keyword matches.

These embeddings will be stored in ChromaDB for similarity-based retrieval.

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks_df["text"].tolist())
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (5, 384)


In [10]:
client = chromadb.PersistentClient(
    path=str(root / "data/chroma_db")
)

collection = client.get_or_create_collection(
    name="dbd_knowledge"
)

In [11]:
collection.upsert(
    ids=[
        f"huntress_{i}"
        for i in range(len(chunks_df))
    ],
    embeddings=embeddings.tolist(),
    documents=chunks_df["text"].tolist(),
    metadatas=[
        {
            "character": row.character,
            "category": row.category,
            "topic": row.topic,
            "source_url": row.source_url
        }
        for row in chunks_df.itertuples()
    ]
)

In [12]:
print("Stored chunks:", collection.count())

Stored chunks: 5


We test the retrieval system using three questions covering lore, powers, and perks.

Each question is converted into an embedding and compared against the stored vectors in ChromaDB. We retrieve the closest matching chunk (n_results=1) and examine its topic and distance.

**Objective:** Verify that the correct information is retrieved at Rank 1.

All three questions retrieved their expected topics, resulting in an initial Hit@1 of 100% (3/3). However, this is a small functionality test rather than a comprehensive evaluation.

In [13]:
questions = [
    "What happened to The Huntress's mother?",
    "How does The Huntress use her Hunting Hatchets?",
    "What does Beast of Prey do?"
]

for question in questions:
    results = collection.query(
        query_embeddings=[model.encode(question).tolist()],
        n_results=1
    )

    print("\nQuestion:", question)
    print("Retrieved:", results["metadatas"][0][0]["topic"])
    print("Distance:", round(results["distances"][0][0], 4))


Question: What happened to The Huntress's mother?
Retrieved: The Huntress’ Lore
Distance: 0.6241

Question: How does The Huntress use her Hunting Hatchets?
Retrieved: Hunting Hatchets
Distance: 0.613

Question: What does Beast of Prey do?
Retrieved: Beast of Prey
Distance: 0.7125
